# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit per-wallet copy weights ``alpha_w`` on **train** per-wallet daily pnl
(3 tiers by Sharpe proxy: top 2x, middle 1x, bottom dropped), pick
hyperparameters on **validation** by sim Sharpe, single **test** pass.
Copy qty is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result.json` + `signal_lab/wallet_scaling_{sim,ci,contrib}.csv`


In [152]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

from lib import DEFAULT_SPLIT, DEFAULT_TAGS
from signal_lab.filters import COPY_DEFAULT
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import candidate_splits_for, load_stage1_data
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_000.0
COST_SEL = 10.0
MAX_LEAD_DAYS = 14  # keep only trades within this many days of contract resolution
ALPHA_MAX_GRID = (2.0, 4.0, 8.0)
TIER_GRID = [(nt, am, amin) for nt in (3, 4, 5) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (0.5, 1.0, 2.0, 4.0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [153]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data(tags=DEFAULT_TAGS, **DEFAULT_SPLIT, max_lead_days=MAX_LEAD_DAYS)
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 2165960
Filtered markets for {'Politics'}: 44273
Loading 16 trade shards...
Total trades loaded: 13,282,697
Unique wallets: 34,344
Date range: 2025-01-01 00:00:59+00:00 -> 2026-08-03 16:01:48+00:00
Lead filter (<= 14d before resolution): 9,865,155 trades
split_data_at_dates: train_end=2026-02-01 val_end=2026-05-31 test_start=2026-06-16
  Train:  3,268,125 trades  (7,815 markets)
  Val:    4,239,985 trades  (7,283 markets)
  Test:   2,357,045 trades  (5,937 markets)
  Total:  9,865,155 trades  (21,035 markets)


/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

invalid value encountered in sqrt



df_full: 9,865,155
  train: 3,268,125  val: 4,239,985  test: 2,357,045


## Copy universe

Candidate wallets = `COPY_DEFAULT` (copy-default filter).

In [154]:
wallets = set(COPY_DEFAULT(wallet_metrics, hold_metrics))
print(f"copy_default wallets: {len(wallets)}")


copy_default wallets: 102


## Share-depth cap

Cap = stage0 Phase 2's per-bucket max copy quantity (`avail_copy_qty`), exported with the processed trades.

In [155]:
splits = candidate_splits_for(df_full, wallets, **DEFAULT_SPLIT)
splits = attach_depth_cap(splits)
del df_full, df_train, df_val, df_test

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr["copyable_qty_5m_100"]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


split_data_at_dates: train_end=2026-02-01 val_end=2026-05-31 test_start=2026-06-16
  Train:     63,294 trades  (5,411 markets)
  Val:       68,046 trades  (4,614 markets)
  Test:      14,308 trades  (1,509 markets)
  Total:    145,648 trades  (11,534 markets)
train: 63,294  trades_capped_by_depth=0.000
val  : 68,046  trades_capped_by_depth=0.000
test : 14,308  trades_capped_by_depth=0.000


## Train per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) with mean/std shrinkage -> Sharpe proxy.

In [156]:
train_daily = wallet_daily_pnl(splits["train"])
st = wallet_stats(train_daily)
print(f"wallets with train daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with train daily series: 102


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0x75049bd489194be19c45c31ed311e556411c9c69,1477.2471,6721.4086,39,0.1791,57612.6361
0xeee9c7196b5cbf3a77ce90259e93cf4218a06e94,1226.2516,4671.2957,16,0.1697,19620.0249
0x858d551d073e9c647c17079ad9021de830201047,1337.1559,6042.9497,25,0.1650,33428.8976
0x169a6428a5e5354fc256b66c5465231f918dc46a,540.8754,1413.7533,16,0.1610,8654.0061
0xfd66d7ed45d7962ad009e669cdaec9319e38fb6d,847.0832,4322.3031,40,0.1586,33883.3287
0x3c593aeb73ebdadbc9ce76d4264a6a2af4011766,333.3615,1934.1267,125,0.1533,41670.1824
0x0c4f5b1295f39cf505679209a22adbafe61c0f33,1308.1435,7103.2719,38,0.1502,49709.4520
0xf72044fb3f98854411a069d9a682e54e4b824599,419.8861,2074.0711,34,0.1429,14276.1285
0x80a0da00fbdc8440b0ef601341f14c3e24795708,308.2201,1003.4124,21,0.1245,6472.6213


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [157]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 26


## Validation grid search

Objective: annualized Sharpe of daily resolution-pnl, fixed $10k budget, 10bps.

In [158]:
sim_rows = []
best_per_scheme = {}
for name, (scheme, alpha_map, params) in schemes.items():
    res = run_sim(splits["val"], alpha_map, COST_SEL)
    row = sim_row(scheme, name, "val", res)
    sim_rows.append(row)
    key = scheme if scheme != "kelly" else "kelly"
    if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
        best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "val"].sort_values("sharpe_daily", ascending=False).head(15)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
3,tier,tier3@2-0,val,9315,102534.5000,0.9262,0.7080,5427.6500,10000.0000
5,tier,tier3@4-0,val,9315,102534.5000,0.9262,0.7080,5427.6500,10000.0000
7,tier,tier3@8-0,val,9315,102534.5000,0.9262,0.7080,5427.6500,10000.0000
19,tier,tier5@8-0,val,11750,107342.7600,0.9643,0.5440,5409.0400,10000.0000
17,tier,tier5@4-0,val,11750,107342.7600,0.9643,0.5440,5409.0400,10000.0000
15,tier,tier5@2-0,val,11750,107342.7600,0.9643,0.5440,5409.0400,10000.0000
13,tier,tier4@8-0,val,11111,95601.2800,0.8620,0.5370,5410.4200,10000.0000
11,tier,tier4@4-0,val,11111,95601.2800,0.8620,0.5370,5410.4200,10000.0000
9,tier,tier4@2-0,val,11111,95601.2800,0.8620,0.5370,5410.4200,10000.0000
1,kelly,kelly@4,val,23144,102289.5100,0.9706,0.4970,5169.5500,10000.0000


In [159]:
print("Selected per scheme (by val Sharpe):")
for key, (name, params, val_sharpe) in best_per_scheme.items():
    print(f"  {key:10s} -> {name:>22s}  val_sharpe={val_sharpe:.3f}")

best_name = max(
    best_per_scheme.values(), key=lambda x: x[2]
)[0]
print(f"\nBest val config overall: {best_name}")


Selected per scheme (by val Sharpe):
  kelly      ->                kelly@2  val_sharpe=0.497
  tier       ->              tier3@2-0  val_sharpe=0.708
  uniform    ->              uniform@1  val_sharpe=0.436
  copy_all   ->               copy_all  val_sharpe=0.436

Best val config overall: tier3@2-0


## Test: single pass per chosen config

One honest test pass for each scheme's val-chosen config (10bps).

In [160]:
for key, (name, params, _val_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, COST_SEL)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(OUT_DIR / "wallet_scaling_sim.csv", index=False)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
26,kelly,kelly@2,test,3465,-3560.7500,-0.3490,-0.9730,8307.5700,10000.0000
28,uniform,uniform@1,test,3333,-1782.0800,-0.1765,-0.9940,8344.0800,10000.0000
29,copy_all,copy_all,test,3333,-1782.0800,-0.1765,-0.9940,8344.0800,10000.0000
27,tier,tier3@2-0,test,2085,-1290.4200,-0.1269,-1.2560,8334.4600,10000.0000


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [161]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    for cost in (0.0, 10.0, 30.0):
        res = run_sim(splits["test"], alpha_map, cost)
        point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
        ci_rows.append({
            "design": name, "cost_bps": cost,
            "pnl": round(res["net_pnl"], 2),
            "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
            "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
            "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
        })

res_all = capital_constrained_sim(splits["test"], "score1", BUDGET, 1.0, cost_bps=COST_SEL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all", "cost_bps": COST_SEL,
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "wallet_scaling_ci.csv", index=False)
ci_df


,design,cost_bps,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@2,0.0000,-3550.5400,-0.3480,-0.9730,-1.3150,2.9750
1,kelly@2,10.0000,-3560.7500,-0.3490,-0.9730,-1.3150,2.9750
2,kelly@2,30.0000,-3581.1500,-0.3510,-0.9730,-1.3150,2.9750
3,tier3@2-0,0.0000,-1280.2500,-0.1259,-1.2560,-2.0540,2.6420
4,tier3@2-0,10.0000,-1290.4200,-0.1269,-1.2560,-2.0540,2.6420
5,tier3@2-0,30.0000,-1310.7600,-0.1289,-1.2560,-2.0540,2.6420
6,uniform@1,0.0000,-1771.9800,-0.1755,-0.9940,-1.8400,3.0940
7,uniform@1,10.0000,-1782.0800,-0.1765,-0.9940,-1.8400,3.0940
8,uniform@1,30.0000,-1802.2800,-0.1785,-0.9940,-1.8400,3.0940
9,copy_all,0.0000,-1771.9800,-0.1755,-0.9940,-1.8400,3.0940


## Test-period exposure & PnL over time

Exposure opens at each BUY (`qty = alpha_w * copyable_qty` capped by `bucket_avail_copy_qty`, at `price`) and closes at contract resolution `last_condition_trade_ts` — only for contracts resolved within the test window, so unresolved exposure stays open. PnL shown twice: attributed at trade time (`dt`) and at contract resolution time (`last_condition_trade_ts`, resolved contracts only).

In [162]:
import plotly.graph_objects as go

test = splits["test"].copy()
alpha_map = schemes[best_name][1]
test["alpha_w"] = test["wallet"].map(alpha_map).fillna(1.0)
test["qty"] = np.clip(test["alpha_w"] * test["copyable_qty_5m_100"], 0.0, test["bucket_avail_copy_qty"])
test["copy_pnl"] = test["copyable_pnl"] / test["copyable_qty_5m_100"].replace(0, np.nan) * test["qty"]

test["res_ts"] = pd.to_datetime(test["last_condition_trade_ts"], utc=True, errors="coerce")
window_end = test["dt"].max()
resolved = test["res_ts"] <= window_end
print(
    f"test trades: {len(test):,}  contracts: {test['condition_id'].nunique():,}  "
    f"resolved by {window_end:%Y-%m-%d}: {int(resolved.sum()):,} trades "
    f"({test.loc[resolved, 'condition_id'].nunique():,} contracts)"
)

open_ev = pd.DataFrame({
    "ev_dt": test["dt"],
    "exposure_delta": test["qty"] * test["price"],
})
close_ev = pd.DataFrame({
    "ev_dt": test.loc[resolved, "res_ts"],
    "exposure_delta": -(test.loc[resolved, "qty"] * test.loc[resolved, "price"]),
})
events = (
    pd.concat([open_ev, close_ev], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
events["exposure"] = events["exposure_delta"].cumsum()

pnl_trade = (
    test[["dt", "copy_pnl"]]
    .rename(columns={"dt": "ev_dt"})
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
pnl_trade["cum_pnl"] = pnl_trade["copy_pnl"].cumsum()

pnl_res = (
    test.loc[resolved, ["res_ts", "copy_pnl"]]
    .rename(columns={"res_ts": "ev_dt"})
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
pnl_res["cum_pnl"] = pnl_res["copy_pnl"].cumsum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=events["ev_dt"], y=events["exposure"], mode="lines", name="exposure"))
fig.add_trace(go.Scatter(
    x=pnl_trade["ev_dt"], y=pnl_trade["cum_pnl"], mode="lines",
    name="cum copyable pnl (trade time)",
))
fig.add_trace(go.Scatter(
    x=pnl_res["ev_dt"], y=pnl_res["cum_pnl"], mode="lines", line=dict(dash="dash"),
    name="cum copyable pnl (resolution time)",
))
fig.update_layout(
    title=f"Test-period exposure & copyable PnL over time — {best_name}",
    xaxis_title="Time",
    yaxis_title="USDC",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()


test trades: 14,308  contracts: 1,509  resolved by 2026-08-03: 14,293 trades (1,504 contracts)


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [163]:
test_daily = wallet_daily_pnl(splits["test"])
test_st = test_daily.groupby("wallet")["copyable_pnl"].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")["copyable_pnl"].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)
contrib.to_csv(OUT_DIR / "wallet_scaling_contrib.csv", index=False)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].head(15)


Spearman(alpha_kelly, wallet test sharpe) = -0.0041  (n=72)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
0,0x02a17a92e6f673129b37d95359c7af628a3cdd72,1.0119,1.0000,2414.5575,12.7240,1.1404
1,0x09343a349bc2b0840c5b81378a229a4ac084f376,0.5457,0.0000,-26.2300,-1.0957,-0.0729
2,0x0a854897a06d4999e5b2dde5693609f1428ffe9d,0.2041,2.0000,-74.2057,0.0000,-0.0012
3,0x0b652d3e55be0e1d50c2f7be39358585c415293e,1.0973,1.0000,-114.6150,-0.7265,-0.0534
4,0x0c4f5b1295f39cf505679209a22adbafe61c0f33,0.6134,2.0000,162.5923,13.5382,0.0033
5,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.9925,0.0000,-360.3209,-0.2028,-0.0631
6,0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,0.5219,0.0000,-14.1384,-3.6117,-0.0492
7,0x0f5830d72a09b6bb1f19793fa8003ef4415e5ff2,1.4930,2.0000,-306.0000,-13.5093,-0.0392
8,0x10b289276b2b69cac7b1bbc58601009b6ee74ceb,0.8805,0.0000,4294.4641,1.9261,2.9597
9,0x140891ee5c1214859198b80f0a5a4880ba57b4f8,0.4402,0.0000,-341.7695,-6.5757,-14.0865


## Save stage 1 result

In [164]:
import json
from datetime import datetime, timezone

best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
best_params = schemes[best_name][2]

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

metadata = {
    "type": "scaled_copy",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_val_sharpe": float(max(best_per_scheme.values(), key=lambda x: x[2])[2]),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = NB_DIR / "stage1_scaled_result.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.


In [165]:
from lib import DEFAULT_TRADES_DIR
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty_5m_100"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty_5m_100>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(DEFAULT_TRADES_DIR.glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 1,509  sampled: 1,000
candidate BUYs (copyable_qty>0) on sample: 5,888
fill tape rows (sampled contracts, both sides): 1,277,199


In [166]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 52,161.49


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,5888,834,0.1400,9178.1500,17.6000,-42983.3400
1,0.9500,5888,1373,0.2300,19253.9100,36.9100,-32907.5800
2,0.9800,5888,2000,0.3400,43777.4300,83.9300,-8384.0700
3,1.0000,5888,5888,1.0000,52161.4900,100.0000,0.0000


In [167]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x80a0da00fbdc8440b0ef601341f14c3e24795708,7180.6000,10691.0000,29731.8000,33021.5000,0.1000,0.2000,0.6000,1.0000
0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,11793.8000,13538.1000,20096.6000,22892.1000,0.1000,0.1000,0.2000,1.0000
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,-3471.1000,1930.4000,4970.5000,5300.8000,0.2000,0.2000,0.3000,1.0000
0x41583f2efc720b8e2682750fffb67f2806fece9f,-351.6000,2115.9000,1984.3000,4104.7000,0.3000,0.4000,0.5000,1.0000
0xf72044fb3f98854411a069d9a682e54e4b824599,0.0000,526.1000,508.8000,2595.0000,0.0000,0.2000,0.2000,1.0000
0xcd6be6693e2bc11150d397caa9e6f7c6744b68f5,1611.6000,1747.1000,2298.3000,2165.7000,0.1000,0.2000,0.3000,1.0000
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,238.2000,226.4000,631.7000,1999.3000,0.2000,0.2000,0.3000,1.0000
0xa4b366ad22fc0d06f1e934ff468e8922431a87b8,-62.5000,-87.2000,-91.9000,374.5000,0.1000,0.2000,0.2000,1.0000
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,164.7000,469.1000,179.6000,311.8000,0.1000,0.2000,0.2000,1.0000


In [168]:
sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")


saved -> signal_lab/price_scale_{sim,summary,wallets}.csv
